# ESM C test (G10) for the cell model

This notebook reads every human protein with **ESM C**, a protein language model, and runs four tests that were
written and declared before any run:

| Test | Question |
|---|---|
| **E1** | Does the whole sequence tell where a protein goes better than the address tags alone? |
| **E2** | Can it find the address of proteins that have **no** known tag? |
| **P1** | Does adding "do these two proteins fit?" from ESM C help the partner finder find a real partner sooner? |
| **S1** | Can it tell which RNA versions (isoforms) change where the protein goes? |

Each test is fit on half the genes and scored once on the other half, with a shuffled control that must fail and
a planted control that must pass.

**How to run**
1. Check the runtime has a GPU: *Runtime → Change runtime type → T4 GPU* (any GPU works).
2. *Runtime → Run all*.
3. When asked, allow Google Drive (so the results are saved to *My Drive / cellbrain_esmc*). Set
   `SAVE_TO_DRIVE = False` below to skip this; the results are also downloaded to your computer.
4. Wait. Reading 42,171 proteins takes about 40–60 min on a T4 (5–10 min on an A100). Keep the tab open.
5. Tell Claude when it is done. Claude reads `G10_results.json` from the Drive folder; or paste the last cell's output.

Fixed version: code commit `3aa4d71316e1`, code id `6c574b9b3f1f2408`. Model `biohub/ESMC-600M` (the declared test).

In [ ]:
# Settings
MODEL = "biohub/ESMC-600M"           # the declared test. "biohub/ESMC-6B" needs an A100/H100 (40 GB+) and counts as exploratory only
SAVE_TO_DRIVE = True                 # save the small results files to My Drive/cellbrain_esmc (so Claude can read them)
KEEP_FINGERPRINTS_IN_DRIVE = False   # also keep the ~100 MB protein fingerprints there, to reuse later without recomputing
COMMIT = "3aa4d71316e156e85689a0540db38797e8271f5c"
EXPECTED_CODE = "6c574b9b3f1f2408"

import torch, subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout or "no GPU found")
assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4 GPU, then Runtime > Run all again."
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

In [ ]:
# Install the library that runs ESM C (about 1 minute)
!pip -q install "transformers==5.24.1"
import transformers; print("transformers", transformers.__version__)

In [ ]:
# Download the test code and its fixed input data from GitHub, and check they are the declared version
import os, hashlib, urllib.request
BASE = f"https://raw.githubusercontent.com/Nikku03/cell/{COMMIT}/colab/esmc/"
os.makedirs("/content/esmc/bundle", exist_ok=True)
for f in ["esmctest.py", "manifest.json", "bundle/proteins.tsv.gz", "bundle/genes.json.gz", "bundle/ppi.tsv.gz", "bundle/shortlist.npz"]:
    urllib.request.urlretrieve(BASE + f, "/content/esmc/" + f)
code_id = hashlib.sha256(open("/content/esmc/esmctest.py", "rb").read() + open("/content/esmc/manifest.json", "rb").read()).hexdigest()[:16]
assert code_id == EXPECTED_CODE, f"code id {code_id} is not the declared {EXPECTED_CODE}"
print("code and data ok:", code_id)

In [ ]:
# Run the test (progress is printed every minute; about 40-60 min on a T4)
!cd /content/esmc && python esmctest.py --model "{MODEL}" --out /content/esmc_out

In [ ]:
# Save and show the results
import json, shutil, os
OUT = "/content/esmc_out"; res_file = f"{OUT}/G10_results.json"; cat_file = f"{OUT}/G10_isoform_catalogue.csv"
assert os.path.exists(res_file), "the test did not finish; look at the output of the cell above"
if SAVE_TO_DRIVE:
    d = "/content/drive/MyDrive/cellbrain_esmc"; os.makedirs(d, exist_ok=True)
    for f in [res_file, cat_file]:
        shutil.copy(f, d)
    if KEEP_FINGERPRINTS_IN_DRIVE:
        for f in os.listdir(OUT):
            if f.startswith("fingerprints_"):
                shutil.copy(f"{OUT}/{f}", d)
    print("saved to My Drive/cellbrain_esmc")
from google.colab import files
files.download(res_file)
res = json.load(open(res_file))
print("\nIf Claude cannot read your Drive, copy everything below this line into the chat:\n")
print(json.dumps(res, default=float))